PROJET : Système de Gestion de Coworking (SQL & Python)
TISNES Francesca - M1 APE - DS2E

Objectif : Démonstration de la migration d'un système de gestion de données non-structurées vers une base de données relationnelle normalisée

1. Justification du Passage au SQL

L'utilisation de fichiers de type Excel présente des limites critiques pour la gestion d'une entreprise :

Redondance des données : Répétition inutile des informations clients à chaque réservation.

Risque d'incohérence : Difficulté de mettre à jour une information globale (ex: adresse du siège social) sur plusieurs centaines de lignes.

Solution proposée : Une structure normalisée en trois tables (entreprises, membres, reservations) liées par des relations d'intégrité référentielle (Clés étrangères)

In [5]:
import sqlite3
import pandas as pd
from faker import Faker

# Connexion au moteur de base de données
connection = sqlite3.connect("coworking_base.db")
cursor = connection.cursor()

# Réinitialisation du schéma (Clean Start)
cursor.execute("DROP TABLE IF EXISTS reservations")
cursor.execute("DROP TABLE IF EXISTS membres")
cursor.execute("DROP TABLE IF EXISTS entreprises")

# Création des tables normalisées
cursor.execute("""
CREATE TABLE entreprises (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nom TEXT NOT NULL,
    secteur TEXT,
    ville TEXT
)
""")

cursor.execute("""
CREATE TABLE membres (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nom_complet TEXT,
    email TEXT,
    entreprise_id INTEGER,
    FOREIGN KEY (entreprise_id) REFERENCES entreprises(id)
)
""")

cursor.execute("""
CREATE TABLE reservations (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    membre_id INTEGER,
    bureau_type TEXT,
    date_resa DATE,
    tarif_heure REAL,
    FOREIGN KEY (membre_id) REFERENCES membres(id)
)
""")

connection.commit()
print("Structure SQL initialisée.")

Structure SQL initialisée.


2. Génération de Données Transactionnelles
Afin de valider la structure, le script utilise la bibliothèque Faker pour simuler un historique de 100 réservations impliquant 10 entreprises et 40 membres.

In [6]:
generator = Faker('fr_FR')

# Insertion des entités 'Entreprises'
for _ in range(10):
    cursor.execute("INSERT INTO entreprises (nom, secteur, ville) VALUES (?, ?, ?)",
                (generator.company(), generator.bs(), generator.city()))

# Insertion des entités 'Membres'
for _ in range(40):
    cursor.execute("INSERT INTO membres (nom_complet, email, entreprise_id) VALUES (?, ?, ?)",
                (generator.name(), generator.email(), generator.random_int(min=1, max=10)))

# Insertion des entités 'Réservations'
options = [('Open Space', 5.0), ('Bureau Privé', 18.0), ('Salle de Réunion', 45.0)]
for _ in range(100):
    type_b, tarif = generator.random_element(options)
    cursor.execute("INSERT INTO reservations (membre_id, bureau_type, date_resa, tarif_heure) VALUES (?, ?, ?, ?)",
                (generator.random_int(min=1, max=40), type_b, generator.date_this_year(), tarif))

connection.commit()
print("Base de données alimentée avec 100 réservations.")

Base de données alimentée avec 100 réservations.


/tmp/ipykernel_4155/3201562048.py:17: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute("INSERT INTO reservations (membre_id, bureau_type, date_resa, tarif_heure) VALUES (?, ?, ?, ?)",


3. Évolutivité et Mise à jour du Schéma
Démonstration de la mise à jour structurelle (ajout de colonne) et de la modification sélective d'attributs.

In [7]:
# Modification structurelle : Ajout de la gestion des badges
cursor.execute("ALTER TABLE membres ADD COLUMN statut_badge TEXT DEFAULT 'Actif'")

# Modification de données : Changement de secteur pour l'ID 1
cursor.execute("UPDATE entreprises SET secteur = 'Tech & Innovation' WHERE id = 1")

connection.commit()
print("Mises à jour structurelles et applicatives terminées.")

Mises à jour structurelles et applicatives terminées.


4. Analyse Business et Extraction de Valeur
Extraction du chiffre d'affaires cumulé par client via des jointures SQL (JOIN) et exportation des résultats au format standard CSV pour analyse complémentaire.

In [8]:
# Requête d'agrégation financière
query = """
SELECT e.nom AS Client, COUNT(r.id) AS Nombre_Reservations, SUM(r.tarif_heure) AS Chiffre_Affaires_HT
FROM entreprises e
JOIN membres m ON e.id = m.entreprise_id
JOIN reservations r ON m.id = r.membre_id
GROUP BY e.nom
ORDER BY Chiffre_Affaires_HT DESC
"""

# Traitement via Pandas
df_finance = pd.read_sql_query(query, connection)

# Exportation automatique
df_finance.to_csv("rapport_activite.csv", index=False)

# Affichage du rapport
df_finance

,Client,Nombre_Reservations,Chiffre_Affaires_HT
0,Rolland,16,398.0
1,Aubry,15,394.0
2,Lombard,17,323.0
3,De Oliveira Valentin et Fils,15,287.0
4,Michaud Hardy S.A.R.L.,12,285.0
5,Pruvost S.A.,9,217.0
6,Teixeira,7,181.0
7,Loiseau SA,6,109.0
8,Andre,2,63.0
9,Cousin Lenoir S.A.R.L.,1,45.0
